In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow.keras import layers, models, losses 
from tensorflow.keras.datasets import mnist 
from tensorflow.keras.metrics import Precision, Recall
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from scipy.ndimage import rotate
from sklearn.model_selection import train_test_split

# fixed seed so the results can be reproduced
SEED = 42
tf.keras.utils.set_random_seed(SEED)  # Python, NumPy and TensorFlow
tf.config.experimental.enable_op_determinism()

In [2]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
# imgs
print(x_train.shape == (60000, 28, 28))
print(x_test.shape == (10000, 28, 28))
# labels 
print(y_train.shape == (60000,))
print(y_test.shape == (10000,))

True
True
True
True


In [3]:
# hold out 6,000 training images for validation (same split in every notebook);
# the test set is used only once, for the final evaluation
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=6000, stratify=y_train, random_state=SEED
)
print(f"train: {x_train.shape[0]:,}  validation: {x_val.shape[0]:,}  test: {x_test.shape[0]:,}")

train: 54,000  validation: 6,000  test: 10,000


In [4]:
angles = [i for i in range(-20, 21, 5)]
NOISE_STD = 20.0
new_x = []
new_y = []

for img, label in zip(x_train, y_train):
    # without augmentation
    new_x.append(img)
    new_y.append(label)

    for angle in angles:
        if angle == 0:
            noise = np.random.normal(loc=0.0, scale=NOISE_STD, size=img.shape)
            img_noised = img + noise
            img_noised = np.clip(img_noised, 0, 255).astype(np.uint8)
            new_x.append(img_noised)
            new_y.append(label)
            continue

        img_rot = rotate(img, angle, axes=(0,1), reshape=False, mode='constant', cval=0.0)
        new_x.append(img_rot)
        new_y.append(label)
        noise = np.random.normal(loc=0.0, scale=NOISE_STD, size=img_rot.shape)
        img_noised = img_rot + noise
        img_noised = np.clip(img_noised, 0, 255).astype(np.uint8)

        new_x.append(img_noised)
        new_y.append(label)

x_train_augmented = np.array(new_x)
y_train_augmented = np.array(new_y)


In [5]:
model = models.Sequential([
    layers.Input((28,28,1)),
    layers.Rescaling(1./255),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(pool_size=(2)),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(pool_size=(2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

In [6]:
model.compile(optimizer='adam', loss=losses.SparseCategoricalCrossentropy(), metrics=['accuracy'],)
model.fit(x=x_train_augmented, y=y_train_augmented, batch_size=64, epochs=1, validation_data=(x_val, y_val))

15188/15188 ━━━━━━━━━━━━━━━━━━━━ 60s 4ms/step - accuracy: 0.9789 - loss: 0.0692 - val_accuracy: 0.9897 - val_loss: 0.0371


In [7]:
model_loss, model_accuracy = model.evaluate(x_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9936 - loss: 0.0213


In [8]:
print(f"Loss: {model_loss}")
print(f"Accuracy: {model_accuracy*100:.2f}")

Loss: 0.02132798545062542
Accuracy: 99.36


In [9]:
predictions = model.predict(x_test)
decisions = np.argmax(predictions, axis=1)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [10]:
print(classification_report(y_test, decisions))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       980
           1       0.99      1.00      1.00      1135
           2       0.99      1.00      0.99      1032
           3       1.00      1.00      1.00      1010
           4       0.99      0.99      0.99       982
           5       0.99      0.99      0.99       892
           6       1.00      0.99      0.99       958
           7       0.99      0.99      0.99      1028
           8       0.99      0.99      0.99       974
           9       0.99      0.99      0.99      1009

    accuracy                           0.99     10000
   macro avg       0.99      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000



In [11]:
cm = confusion_matrix(y_test, decisions)
df_cm = pd.DataFrame(cm)
df_cm.index.name = "Real"
df_cm.columns.name = "Predicted"
df_cm

Predicted,0,1,2,3,4,5,6,7,8,9
Real,,,,,,,,,,
0,977,0,0,0,1,0,1,1,0,0
1,0,1134,0,1,0,0,0,0,0,0
2,1,1,1027,0,0,0,0,3,0,0
3,0,1,0,1005,0,3,0,0,1,0
4,0,0,0,0,973,0,0,1,2,6
5,0,0,0,3,0,887,2,0,0,0
6,2,3,0,0,2,1,950,0,0,0
7,0,3,4,0,0,0,0,1021,0,0
8,0,0,2,1,0,0,0,1,966,4


In [12]:
model.save('../models/digit-recognizer-with-augmentation.keras')